# Task B · SQLite queries and CRUD

Prepare a database copy, run 9 queries, and verify the addition, deletion, change, and query of 8 tables

### Mount Google Drive and prepare the test database

- `drive.mount("/content/drive")` makes Google Drive available in Colab. `COLAB_PROJECT` points to the project directory. The setup checks the current directory, its parent, and the configured Drive directory for both `src/queries.py` and `data/db/air_quality.db`. If the project is not found, it checks the configured Drive location again and reports a missing path if needed.
- `sys.path.insert(...)` makes the project modules available to Python. `queries` supplies the saved SQL, `crud` supplies the record operations, and `get_connection()` opens a SQLite database.
- `TemporaryDirectory()` creates a workspace for the test database. The suffix `mode=ro` opens the original in read-only mode. `source.backup(copy)` copies it to `demo_db`, and `seed_saved_queries()` registers the nine queries in that copy.
- `run()` takes a saved query code and named inputs, then returns a list of rows as dictionaries. `show()` prints the requested fields from those rows. Look for `Registered 9 saved queries on the demo copy.` before running the examples.

In [ ]:
from contextlib import closing
import json
from pathlib import Path
import sqlite3
import sys
from tempfile import TemporaryDirectory

COLAB_PROJECT = Path("/content/drive/MyDrive/groupby_team_projecct")

def has_project(path):
    return (path / "src/queries.py").is_file() and (path / "data/db/air_quality.db").is_file()

candidates = (Path.cwd(), Path.cwd().parent, COLAB_PROJECT)
project = next((path for path in candidates if has_project(path)), None)
if project is None:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise FileNotFoundError("Open the notebook from the Groupby project or its src directory.") from exc
    drive.mount("/content/drive")
    project = COLAB_PROJECT
if not has_project(project):
    raise FileNotFoundError(f"Set COLAB_PROJECT to the folder containing src/ and data/: {project}")

sys.path.insert(0, str(project.resolve()))
from src import crud
from src.queries import run_saved_query, seed_saved_queries
from src.schema import get_connection

demo_directory = TemporaryDirectory(prefix="groupby_task_b_")
demo_db = Path(demo_directory.name) / "task_b_demo.db"
original_uri = (project / "data/db/air_quality.db").resolve().as_uri() + "?mode=ro"
with closing(sqlite3.connect(original_uri, uri=True)) as source, closing(get_connection(demo_db)) as copy:
    source.backup(copy)

print(f"Registered {seed_saved_queries(db_path=demo_db)} saved queries on the demo copy.")

def run(code, **params):
    return run_saved_query(code, params, db_path=demo_db)

def show(rows, *columns):
    for row in rows:
        print({column: row[column] for column in columns})

Mounted at /content/drive
Registered 9 saved queries on the demo copy.


### Query interface for Tasks C and D

Pass a query code and a dictionary of inputs to `run_saved_query()`. Each returned dictionary represents one result row. Use `YYYY-MM-DD` for a date and `YYYY-MM` for a month. A date range includes both endpoints.

| Query | Inputs | What each result row represents |
| --- | --- | --- |
| Q01 | `station_name`, `obs_date` | A station's daily AQHI, coverage, pollutant, and weather summary |
| Q02 | `station_name`, `obs_date` | A risk category with its hours and percentage of valid hours |
| Q03 | `station_a`, `station_b`, `obs_date` | A station in the comparison, in the order requested |
| Q04 | `station_name`, `start_date`, `end_date` | One day's pollution and weather context |
| Q05 | `obs_date` | A district summary or a roadside station marker, identified by `row_type` |
| Q06 | `station_name`, `month` | A monthly summary for the selected station |
| Q07 | `station_name`, `start_date`, `end_date` | A day on which AQHI reached 7 or higher |
| Q08 | `obs_date` | A territory-wide daily summary based on general stations |
| Q09 | No parameters | A count of generated reports for one template and creation month |

Use `max_aqhi` for display and `max_rank` for ordering. AQHI means use `numeric_value`. Each pollutant mean has its own measurement count, such as `valid_pm25_hours`. An empty result list means no matching records. For district map rows, check the coverage fields and peak rank because the output retains districts with missing readings.

## 任务 C 和 D 的查询接口

先运行 seed_saved_queries，再用 run_saved_query(query_code, params) 读取。返回值为字典列表。日期用 YYYY-MM-DD，日期区间含首尾，月份用 YYYY-MM。

| query_code | params 字典的键 | 返回粒度和常用字段 |
| --- | --- | --- |
| Q01_daily_briefing | station_name, obs_date | 一站一天。valid_hours, min_rank, max_rank, max_aqhi, peak_hours 和污染物、气象字段 |
| Q02_category_breakdown | station_name, obs_date | 一行对应一个风险等级，固定顺序从 Low 到 Serious。category, hours, percentage, valid_hours, missing_hours, substitute_hours |
| Q03_station_comparison | station_a, station_b, obs_date | 每个有记录的站一行，按输入顺序。station_name, station_type, max_aqhi, 五类小时数和污染物字段 |
| Q04_weather_pollution | station_name, start_date, end_date | 按日返回，字段沿用 Q01。mean_temp_c, mean_wind_kmh, rainfall_mm 等气象字段来自当日记录 |
| Q05_district_map | obs_date | row_type=district 的 18 行使用一般站日最高值。row_type=roadside 的 3 行含独立站名与坐标。max_rank 为空的区留灰 |
| Q06_monthly_summary | station_name, month | 一站一月。month, valid_hours, missing_hours, max_aqhi, 五类小时数和污染物字段 |
| Q07_high_risk_bulletin | station_name, start_date, end_date | 每个有 AQHI ≥ 7 的日期一行。high_or_above_hours, high_risk_hours, first_high_hour, last_high_hour, max_aqhi |
| Q08_news_flash | obs_date | 一般站全港日摘要。peak_stations, peak_hours, peak_station_hours, high_risk_hours。high_or_above_hours 统计不同钟点，high_station_hours 统计站点小时 |
| Q09_report_history | 空字典或省略 | 按模板及生成月份汇总。template_name, month, report_count, station_count, reports_without_station |

- max_rank 用于排序，max_aqhi 用于展示（包含 10+）。avg_aqhi 只使用 numeric_value，numeric_aqhi_hours 给出分母。
- 污染物沿用 avg_pm25、valid_pm25_hours、pm25_unit 这种命名。相同命名适用于 pm10、o3、no2、so2、co。逐日和逐月查询保留污染物缺测时的 AQHI 小时。
- [] 表示没有符合条件的记录。若一组小时全部缺测，valid_hours 为 0，峰值为空。Q05 即使日期无数据也保留地图行，可用 valid_hours 和 max_rank 判断覆盖。
- 跨越 2025-03-22 的解释放在任务 C 的模板层。日气象是同日背景。

### Daily briefing, risk categories, station comparison, and weather

- **Q01** checks two dates for `Central/Western`. On 1 April, the demo prints the daily AQHI range, all peak hours, the valid-hour count, and mean PM2.5. On 14 April, it shows how the query displays `10+`. The demonstration data for that date contains 24 valid AQHI hours and 7 valid PM2.5 measurements.
- **Q02** groups the valid hours on 1 April by risk category. The `hours` column is the count in that category. The `percentage` column uses all valid AQHI hours on that date as its denominator. For example, Low accounts for 19 hours, or 79.2%.
- **Q03** requests the `Central/Western` general station and the `Central` roadside station on the same date. Compare the maximum AQHI, mean PM2.5, and PM2.5 measurement count in their separate rows.
- **Q04** requests a series for `Sha Tin` from 21 to 23 March inclusive. Each row shows the date and its maximum AQHI beside that day's mean temperature, mean wind speed, and rainfall.

In [ ]:
daily = run("Q01_daily_briefing", station_name="Central/Western", obs_date="2025-04-01")
show(daily, "station_name", "valid_hours", "min_aqhi", "max_aqhi", "peak_hours", "avg_pm25")
serious = run("Q01_daily_briefing", station_name="Central/Western", obs_date="2025-04-14")
show(serious, "max_aqhi", "peak_hours", "valid_hours", "valid_pm25_hours")

categories = run("Q02_category_breakdown", station_name="Central/Western", obs_date="2025-04-01")
show(categories, "category", "hours", "percentage")

comparison = run(
    "Q03_station_comparison", station_a="Central/Western", station_b="Central", obs_date="2025-04-01"
)
show(comparison, "station_name", "station_type", "max_aqhi", "avg_pm25", "valid_pm25_hours")

weather = run("Q04_weather_pollution", station_name="Sha Tin", start_date="2025-03-21", end_date="2025-03-23")
show(weather, "obs_date", "max_aqhi", "mean_temp_c", "mean_wind_kmh", "rainfall_mm")

{'station_name': 'Central/Western', 'valid_hours': 24, 'min_aqhi': '2', 'max_aqhi': '4', 'peak_hours': '17:00, 18:00, 19:00, 20:00, 21:00', 'avg_pm25': 8.92}
{'max_aqhi': '10+', 'peak_hours': '18:00, 19:00, 20:00', 'valid_hours': 24, 'valid_pm25_hours': 7}
{'category': 'Low', 'hours': 19, 'percentage': 79.2}
{'category': 'Moderate', 'hours': 5, 'percentage': 20.8}
{'category': 'High', 'hours': 0, 'percentage': 0.0}
{'category': 'Very high', 'hours': 0, 'percentage': 0.0}
{'category': 'Serious', 'hours': 0, 'percentage': 0.0}
{'station_name': 'Central/Western', 'station_type': 'General', 'max_aqhi': '4', 'avg_pm25': 8.92, 'valid_pm25_hours': 24}
{'station_name': 'Central', 'station_type': 'Roadside', 'max_aqhi': '4', 'avg_pm25': 12.5, 'valid_pm25_hours': 22}
{'obs_date': '2025-03-21', 'max_aqhi': '7', 'mean_temp_c': 20.5, 'mean_wind_kmh': 7.1, 'rainfall_mm': 0.0}
{'obs_date': '2025-03-22', 'max_aqhi': '6', 'mean_temp_c': 21.2, 'mean_wind_kmh': 5.1, 'rainfall_mm': 0.0}
{'obs_date': '2025

### District map, monthly summary, high-risk bulletin, and news flash

- **Q05** returns map data for 1 April. The list comprehensions use `row_type` to split the result into 18 districts and 3 roadside stations. The district layer takes its peak from general stations. A missing `max_rank` identifies a district to leave grey. The roadside layer uses the individual stations' latitude and longitude.
- **Q06** summarizes April 2025 for `Central/Western`. The printed fields give the valid AQHI hours, monthly maximum, mean PM2.5, and the number of measurements used in that mean.
- **Q07** returns dates with AQHI of 7 or higher for the selected station and range. The example selects 14 April alone and displays the number and times of qualifying hours.
- **Q08** summarizes peaks across general stations on 1 April. `peak_stations` and `peak_hours` list where and when the maximum occurred. `high_or_above_hours` counts distinct clock hours when any general station reached the threshold.

In [ ]:
map_rows = run("Q05_district_map", obs_date="2025-04-01")
district_rows = [row for row in map_rows if row["row_type"] == "district"]
roadside_rows = [row for row in map_rows if row["row_type"] == "roadside"]
print(f"Map: {len(district_rows)} districts and {len(roadside_rows)} roadside stations")
show([row for row in district_rows if row["max_rank"] is None], "district_name", "general_station_count")
show(roadside_rows, "station_name", "latitude", "longitude", "max_aqhi")

monthly = run("Q06_monthly_summary", station_name="Central/Western", month="2025-04")
show(monthly, "station_name", "month", "valid_hours", "max_aqhi", "avg_pm25", "valid_pm25_hours")

bulletin = run(
    "Q07_high_risk_bulletin", station_name="Central/Western", start_date="2025-04-14", end_date="2025-04-14"
)
show(bulletin, "obs_date", "max_aqhi", "high_or_above_hours", "high_risk_hours")

flash = run("Q08_news_flash", obs_date="2025-04-01")
show(flash, "max_aqhi", "peak_stations", "peak_hours", "high_or_above_hours")

Map: 18 districts and 3 roadside stations
{'district_name': 'Kowloon City', 'general_station_count': 0}
{'district_name': 'Wan Chai', 'general_station_count': 0}
{'district_name': 'Wong Tai Sin', 'general_station_count': 0}
{'district_name': 'Yau Tsim Mong', 'general_station_count': 0}
{'station_name': 'Central', 'latitude': 22.2818145, 'longitude': 114.15812743, 'max_aqhi': '4'}
{'station_name': 'Causeway Bay', 'latitude': 22.28013296, 'longitude': 114.18509009, 'max_aqhi': '4'}
{'station_name': 'Mong Kok', 'latitude': 22.32261115, 'longitude': 114.16827176, 'max_aqhi': '4'}
{'station_name': 'Central/Western', 'month': '2025-04', 'valid_hours': 720, 'max_aqhi': '10+', 'avg_pm25': 17.25, 'valid_pm25_hours': 682}
{'obs_date': '2025-04-14', 'max_aqhi': '10+', 'high_or_above_hours': 22, 'high_risk_hours': '03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00, 24:00'}
{'max_aqhi': '5', 'peak_statio

### What the CRUD functions return

CRUD stands for create, read, update, and delete. A `create_*()` function returns the new primary key. A `get_*()` function returns a dictionary for a matching record or `None` when there is no match. The update and delete functions return a Boolean to indicate whether they matched a record.

AQHI changes entered through `raw_value` keep the derived risk and missing-value fields in sync. Template placeholders and report parameters are stored as JSON text, so the examples use `json.dumps()` to convert Python lists and dictionaries to strings.

## 8 张表的增删改查

create_* 返回主键，get_* 返回字典或 None。update_* 和 delete_* 返回是否匹配到记录。
AQHI 表通过 raw_value 同步更新等级和缺测标记。模板占位符与报告参数用 JSON 文本保存。

### Define the CRUD checks and look up existing IDs

`roundtrip()` receives the four functions for one table. It creates a record from `fields`, reads that record by its new `row_id`, and applies `changes`. It then reads the record again to verify the changed values. Finally, it deletes the record and verifies that a new lookup returns `None`. Each `assert` stops the test if its condition fails. The success message appears after all checks pass.

In `create(db_path=demo_db, **fields)`, `**fields` supplies the entries in a dictionary as named arguments. The bottom of this cell reads `station_id` from the earlier Q01 result and looks up the corresponding `district_id` and the saved Q01 `query_id`. Later inserts use these existing IDs to satisfy foreign-key constraints.

In [ ]:
def roundtrip(table, create, get, update, delete, fields, changes):
    row_id = create(db_path=demo_db, **fields)
    original = get(row_id, db_path=demo_db)
    assert original is not None
    assert update(row_id, db_path=demo_db, **changes)
    changed = get(row_id, db_path=demo_db)
    assert all(changed[name] == value for name, value in changes.items())
    assert delete(row_id, db_path=demo_db)
    assert get(row_id, db_path=demo_db) is None
    print(f"{table}: created, read, updated and deleted row {row_id}")

station_id = daily[0]["station_id"]
with closing(get_connection(demo_db)) as conn:
    district_id = conn.execute("SELECT district_id FROM stations WHERE station_id = ?", (station_id,)).fetchone()[0]
    query_id = conn.execute(
        "SELECT query_id FROM saved_queries WHERE query_code = ?", ("Q01_daily_briefing",)
    ).fetchone()[0]

### Test the district and station tables

The first call creates a sample district and checks that its area can be updated to `2.5` square kilometres. The second creates a general monitoring station and checks that its latitude can be updated to `22.28`. The station uses the existing `district_id` looked up in the preceding cell. Each call checks the new value and removes its sample record from the test database.

In [ ]:
roundtrip(
    "districts", crud.create_district, crud.get_district, crud.update_district, crud.delete_district,
    {"district_code": "TASKB_DEMO", "english_name": "Task B Demo District", "region": "Kowloon"},
    {"area_sqkm": 2.5},
)
roundtrip(
    "stations", crud.create_station, crud.get_station, crud.update_station, crud.delete_station,
    {"code": "TASKB_DEMO", "name": "Task B Demo Station", "station_type": "General", "district_id": district_id},
    {"latitude": 22.28},
)

districts: created, read, updated and deleted row 19
stations: created, read, updated and deleted row 19


### Test the hourly AQHI, pollutant, and daily weather tables

- The hourly tests use the existing `station_id`, the demonstration date `2099-01-01`, and hour `24`. The AQHI test starts with `4*`, a substitute reading, and changes it to `10+`. The check covers the display value, ranking value of `11`, empty numeric value, and cleared substitute flag.
- The pollutant test changes PM2.5 from `8.0` to `9.5`, with its unit stored as `ug/m3`. The weather test creates a daily record with a mean temperature of `19.0` °C and updates rainfall to `1.5` mm.
- Each call verifies the saved change and deletes its sample row.

In [ ]:
roundtrip(
    "hourly_aqhi", crud.create_hourly_aqhi, crud.get_hourly_aqhi, crud.update_hourly_aqhi, crud.delete_hourly_aqhi,
    {"station_id": station_id, "obs_date": "2099-01-01", "obs_hour": 24, "raw_value": "4*"},
    {"raw_value": "10+", "numeric_value": None, "sort_rank": 11, "is_substitute": 0},
)
roundtrip(
    "hourly_pollutants",
    crud.create_hourly_pollutant, crud.get_hourly_pollutant, crud.update_hourly_pollutant, crud.delete_hourly_pollutant,
    {"station_id": station_id, "obs_date": "2099-01-01", "obs_hour": 24, "pm25": 8.0, "pm25_unit": "ug/m3"},
    {"pm25": 9.5},
)
roundtrip(
    "daily_weather",
    crud.create_daily_weather, crud.get_daily_weather, crud.update_daily_weather, crud.delete_daily_weather,
    {"obs_date": "2099-01-01", "mean_temp_c": 19.0},
    {"rainfall_mm": 1.5},
)

hourly_aqhi: created, read, updated and deleted row 157681
hourly_pollutants: created, read, updated and deleted row 157681
daily_weather: created, read, updated and deleted row 366


### Test the saved queries and report template tables

The saved-query test creates a simple query with code `TASKB_DEMO` and an empty parameter list. The update changes its description. The CRUD check reads that change and then deletes the sample entry.

`template_fields` defines a territory-wide template that refers to the registered Q01 query through `query_id`. The placeholder `{max_aqhi}` appears in the plain-text, Markdown, and HTML versions of the template. The test creates the row, updates the version to `1.1`, verifies that value, and deletes the row. The field dictionary is reused in the next cell.

In [ ]:
roundtrip(
    "saved_queries", crud.create_saved_query, crud.get_saved_query, crud.update_saved_query, crud.delete_saved_query,
    {"query_code": "TASKB_DEMO", "name": "Demo query", "sql_text": "SELECT 1 AS value", "parameter_names": "[]"},
    {"description": "Used in the Task B notebook"},
)

template_fields = {
    "query_id": query_id, "name": "Task B Demo Template", "scope": "territory",
    "required_variables": json.dumps(["max_aqhi"]), "text_template": "AQHI {max_aqhi}",
    "markdown_template": "**AQHI {max_aqhi}**", "html_template": "<p>AQHI {max_aqhi}</p>",
}
roundtrip(
    "report_templates",
    crud.create_report_template, crud.get_report_template, crud.update_report_template, crud.delete_report_template,
    template_fields, {"version": "1.1"},
)

saved_queries: created, read, updated and deleted row 10
report_templates: created, read, updated and deleted row 1


### Test a generated report and the template's foreign-key protection

This cell uses the template fields to create a new template and a sample report that references it. `station_id=None` leaves the station field empty for the area-wide example. The next two checks confirm that the report's title can be updated and read back.

Q09 then looks up report history for that template. The sample created in April 2025 should contribute one report and one report without a station. The cell tries to delete the template while the report still references it. The caught `ReferencedRowError` is the expected evidence that the relationship is protected. Finally, it deletes the report first and then the template, and prints the success message for `generated_reports`.

In [ ]:
template_id = crud.create_report_template(db_path=demo_db, **template_fields)
report_id = crud.create_generated_report(
    db_path=demo_db,
    template_id=template_id, station_id=None, title="Demo briefing",
    parameter_values=json.dumps({"obs_date": "2025-04-14"}), reporting_period="2025-04-14",
    text_output="AQHI 10+", markdown_output="**AQHI 10+**", html_output="<p>AQHI 10+</p>",
    created_at="2025-04-15 09:00:00",
)
assert crud.get_generated_report(report_id, db_path=demo_db)["station_id"] is None
assert crud.update_generated_report(report_id, db_path=demo_db, title="Revised demo briefing")
assert crud.get_generated_report(report_id, db_path=demo_db)["title"] == "Revised demo briefing"

history = run("Q09_report_history")
show([row for row in history if row["template_id"] == template_id],
     "template_name", "month", "report_count", "reports_without_station")

try:
    crud.delete_report_template(template_id, db_path=demo_db)
except crud.ReferencedRowError as exc:
    print(exc)
else:
    raise AssertionError("Expected foreign-key protection.")

assert crud.delete_generated_report(report_id, db_path=demo_db)
assert crud.get_generated_report(report_id, db_path=demo_db) is None
assert crud.delete_report_template(template_id, db_path=demo_db)
print("generated_reports: created, read, updated and deleted the demo report")

{'template_name': 'Task B Demo Template', 'month': '2025-04', 'report_count': 1, 'reports_without_station': 1}
Cannot delete report_templates row 2. Referenced by generated_reports.template_id (1 row(s)).
generated_reports: created, read, updated and deleted the demo report


## Group report material

The SQLite schema contains eight tables. Districts contain monitoring stations, and each station has separate hourly AQHI and pollutant tables with unique station, date and hour keys. Daily weather joins by date. Saved queries feed report templates, which in turn link to generated reports. A generated report can leave the station reference empty when it covers a wider area. The query library uses parameterized SQL to calculate coverage, risk distribution and peaks. It preserves AQHI rows when measurements are missing and reports the sample size of each pollutant mean. The CRUD functions check references before deleting parent records.

For a short demonstration, start with the ER diagram, then show Q01 on 14 April 2025 and the blocked template deletion above.